# Importações e instalações



In [ ]:
!pip -q install osmnx networkx geopandas matplotlib pandas numpy scipy

In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from IPython.display import display, HTML

 # Parâmetros Experimentais

In [ ]:
# Configura OSMnx
ox.settings.log_console = True
ox.settings.use_cache = True
ox.settings.timeout = 300

# Configura matplotlib para o Colab
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

# Pipeline de A* + MST


In [ ]:
class CityPOIAnalyzer:
    def __init__(self):
        self.results = []
        # Cria diretório no Colab
        os.makedirs('/content/results/graphs', exist_ok=True)

    def get_police_stations(self, place):
        # Obtém apenas delegacias de polícia
        print(f"Buscando delegacias em {place.split(',')[0]}...")

        try:
            tags = {'amenity': 'police'}
            pois = ox.features.features_from_place(place, tags=tags)
            police_points = []

            for idx, row in pois.iterrows():
                if row.geometry.geom_type == 'Point':
                    police_points.append((row.geometry.y, row.geometry.x))
                else:
                    police_points.append((row.geometry.centroid.y, row.geometry.centroid.x))

            print(f" {len(police_points)} delegacias encontradas")

            # Mostra algumas delegacias encontradas
            if len(police_points) > 0:
                print(f" Primeiras coordenadas: {police_points[:3]}")

            return police_points

        except Exception as e:
            print(f" Erro ao buscar delegacias: {e}")
            return []

    def find_nearest_nodes(self, G, poi_points):
        # Encontra os nós mais próximos para cada delegacia
        print(f"  Encontrando nós para {len(poi_points)} delegacias...")

        # nearest_nodes espera: (graph, X, Y) onde X=longitude, Y=latitude
        latitudes = [p[0] for p in poi_points]  # Y = latitude
        longitudes = [p[1] for p in poi_points]  # X = longitude

        nearest_nodes = ox.distance.nearest_nodes(G, longitudes, latitudes)
        unique_nodes = list(set(nearest_nodes))

        print(f" {len(unique_nodes)} nós únicos encontrados")

        # Debug: mostrar se há nós duplicados
        if len(unique_nodes) < len(nearest_nodes):
            print(f" {len(nearest_nodes) - len(unique_nodes)} nós duplicados removidos")

        return unique_nodes

    def process_city(self, city_name):
        # Processa uma cidade: baixa grafo, encontra delegacias, calcula rotas A* e MST
        print(f"\n{'='*60}")
        print(f" Processando {city_name}")
        print(f"{'='*60}")

        try:
            # Baixa e preparar o grafo viário
            print("1. Baixando grafo viário...")
            G = ox.graph_from_place(city_name, network_type='drive')
            print(f" Grafo baixado: {len(G.nodes())} nós, {len(G.edges())} arestas")

            # Obtém delegacias
            print("2.  Buscando delegacias...")
            police_points = self.get_police_stations(city_name)

            if len(police_points) < 2:
                print(f" Delegacias insuficientes ({len(police_points)}) em {city_name}. Pulando...")
                return None

            # Encontra nós mais próximos
            print("3. Encontrando nós mais próximos...")
            police_nodes = self.find_nearest_nodes(G, police_points)

            if len(police_nodes) < 2:
                print(f" Nós insuficientes ({len(police_nodes)}) para calcular MST. Pulando...")
                return None

            # Projeta o grafo para métrica UTM
            print("4.  Projetando grafo para UTM...")
            G_proj = ox.projection.project_graph(G)

            # Calcula grafo completo com A* entre delegacias
            print("5. Calculando rotas A* entre delegacias...")
            complete_graph = nx.Graph()
            routes_calculated = 0
            failed_routes = 0

            total_pairs = len(police_nodes) * (len(police_nodes) - 1) // 2
            print(f" Total de pares a calcular: {total_pairs}")

            # Barra de progresso simples
            from tqdm import tqdm
            total_combinations = sum(1 for i in range(len(police_nodes)) for j in range(i+1, len(police_nodes)))

            pbar = tqdm(total=total_combinations, desc=f"Rotas {city_name.split(',')[0]}")

            for i in range(len(police_nodes)):
                complete_graph.add_node(police_nodes[i])
                for j in range(i + 1, len(police_nodes)):
                    try:
                        # Calcula rota A* com heurística euclidiana
                        path = nx.astar_path(
                            G_proj,
                            police_nodes[i],
                            police_nodes[j],
                            heuristic=lambda u, v: ox.distance.euclidean(
                                G_proj.nodes[u]['y'], G_proj.nodes[u]['x'],
                                G_proj.nodes[v]['y'], G_proj.nodes[v]['x']
                            ),
                            weight='length'
                        )

                        # Calcula comprimento total da rota
                        path_length = sum(
                            G_proj[path[k]][path[k+1]][0]['length']
                            for k in range(len(path)-1)
                        )

                        complete_graph.add_edge(
                            police_nodes[i],
                            police_nodes[j],
                            weight=path_length,
                            path=path
                        )
                        routes_calculated += 1

                    except Exception as e:
                        failed_routes += 1
                    finally:
                        pbar.update(1)

            pbar.close()
            print(f" Rotas calculadas: {routes_calculated} (falhas: {failed_routes})")

            # Calcular MST
            if complete_graph.number_of_edges() > 0:
                mst_edges = list(nx.minimum_spanning_edges(complete_graph, data=True))
                total_mst_length = sum(edge[2]['weight'] for edge in mst_edges)

                # Coletar rotas da MST
                mst_routes = []
                for u, v, data in mst_edges:
                    if 'path' in data:
                        mst_routes.append(data['path'])

                print(f"6. MST calculada: {total_mst_length/1000:.2f} km, {len(mst_routes)} rotas")
                print(f"{city_name.split(',')[0]} processada com sucesso!")

                return {
                    'city': city_name,
                    'graph': G_proj,
                    'police_nodes': police_nodes,
                    'complete_graph': complete_graph,
                    'mst_edges': mst_edges,
                    'mst_routes': mst_routes,
                    'total_mst_length': total_mst_length,
                    'num_police_stations': len(police_points),
                    'num_unique_nodes': len(police_nodes),
                    'num_routes': len(mst_routes)
                }
            else:
                print("Nenhuma rota válida calculada")
                return None

        except Exception as e:
            print(f"Erro processando {city_name}: {e}")
            import traceback
            traceback.print_exc()
            return None

    def analyze_all_cities(self, cities):
        # Analisa todas as cidades
        successful_cities = 0

        print("INICIANDO ANÁLISE COMPLETA")
        print("=" * 50)

        for i, city in enumerate(cities, 1):
            print(f"\n [{i}/{len(cities)}] Próxima cidade: {city}")
            result = self.process_city(city)
            if result:
                self.results.append(result)
                successful_cities += 1

        print(f"\n{'='*60}")
        print(f"RESUMO: {successful_cities} de {len(cities)} cidades processadas com sucesso")
        print(f"{'='*60}")

        return self.generate_report()

    def generate_report(self):
        # Gera relatório comparativo
        if not self.results:
            print(" Nenhum resultado válido para gerar relatório")
            return None

        report_data = []

        for result in self.results:
            num_edges_mst = len(result['mst_edges'])
            avg_edge_length = result['total_mst_length'] / num_edges_mst if num_edges_mst > 0 else 0

            report_data.append({
                'Cidade': result['city'].split(',')[0],
                'Total MST (km)': round(result['total_mst_length'] / 1000, 2),
                'Delegacias': result['num_police_stations'],
                'Nós Únicos': result['num_unique_nodes'],
                'Arestas na MST': num_edges_mst,
                'Média por Aresta (km)': round(avg_edge_length / 1000, 2),
                'Média por Delegacia (km)': round(result['total_mst_length'] / result['num_police_stations'] / 1000, 2)
            })

        df = pd.DataFrame(report_data)

        # Estilizar a tabela para melhor visualização
        styled_df = df.style\
            .background_gradient(subset=['Total MST (km)'], cmap='YlOrRd')\
            .background_gradient(subset=['Delegacias'], cmap='Blues')\
            .format({
                'Total MST (km)': '{:.2f}',
                'Média por Aresta (km)': '{:.2f}',
                'Média por Delegacia (km)': '{:.2f}'
            })

        print("\n" + "="*80)
        print(" RELATÓRIO COMPARATIVO - MST ENTRE DELEGACIAS")
        print("="*80)
        display(styled_df)

        # Salvar resultados
        df.to_csv('/content/results/comparative_analysis.csv', index=False)

        if len(report_data) > 0:
            total_km = sum(item['Total MST (km)'] for item in report_data)
            avg_km = total_km / len(report_data)
            max_city = max(report_data, key=lambda x: x['Total MST (km)'])
            min_city = min(report_data, key=lambda x: x['Total MST (km)'])

            print(f"\n ESTATÍSTICAS GERAIS:")
            print(f"    Distância total MST: {total_km:.2f} km")
            print(f"    Média por cidade: {avg_km:.2f} km")
            print(f"    Maior MST: {max_city['Cidade']} ({max_city['Total MST (km)']} km)")
            print(f"    Menor MST: {min_city['Cidade']} ({min_city['Total MST (km)']} km)")

        return df

    def plot_results(self):
        # Plota resultados para cada cidade
        if not self.results:
            print(" Nenhum resultado para plotar")
            return

        print(f"\n Gerando gráficos para {len(self.results)} cidades...")

        for result in self.results:
            try:
                city_name_short = result['city'].split(',')[0]
                print(f"  Plotando {city_name_short}...")

                # Criar figura maior para melhor visualização
                fig, ax = plt.subplots(figsize=(15, 12))

                # Plotar o grafo base
                ox.plot_graph(
                    result['graph'],
                    ax=ax,
                    node_size=0,
                    edge_color="lightgray",
                    edge_linewidth=0.3,
                    show=False,
                    close=False
                )

                # Plotar rotas da MST em vermelho
                for route in result['mst_routes']:
                    x = [result['graph'].nodes[n]['x'] for n in route]
                    y = [result['graph'].nodes[n]['y'] for n in route]
                    ax.plot(x, y, color='red', linewidth=3, zorder=4, alpha=0.8)

                # Plotar delegacias em azul
                police_x = [result['graph'].nodes[n]['x'] for n in result['police_nodes']]
                police_y = [result['graph'].nodes[n]['y'] for n in result['police_nodes']]
                ax.scatter(police_x, police_y, c='blue', s=100, zorder=5,
                          edgecolor='black', linewidth=1.5, label='Delegacias')

                plt.title(f" MST entre Delegacias - {city_name_short}\n"
                         f" Total: {result['total_mst_length']/1000:.2f} km | "
                         f" Delegacias: {result['num_police_stations']}",
                         fontsize=14, pad=20)
                plt.legend(fontsize=12)

                plt.tight_layout()
                filename = f'/content/results/graphs/mst_{city_name_short.lower().replace(" ", "_")}.png'
                plt.savefig(filename, dpi=150, bbox_inches='tight')
                plt.close()

                print(f" Gráfico salvo: {filename}")

            except Exception as e:
                print(f" Erro ao plotar {result['city']}: {e}")

# Cidades para análise
CITIES = [
    "Maceió, Alagoas, Brazil",
    "Salvador, Bahia, Brazil",
    "Fortaleza, Ceará, Brazil",
    "São Luís, Maranhão, Brazil",
    "João Pessoa, Paraíba, Brazil",
    "Recife, Pernambuco, Brazil",
    "Teresina, Piauí, Brazil",
    "Natal, Rio Grande do Norte, Brazil",
    "Aracaju, Sergipe, Brazil"
]

# Executar análise
if __name__ == "__main__":
    print(" Configurando ambiente...")

    # Instalar tqdm para barra de progresso
    try:
        from tqdm import tqdm
    except:
        !pip install tqdm
        from tqdm import tqdm

    analyzer = CityPOIAnalyzer()

    print(" INICIANDO ANÁLISE DE MST ENTRE DELEGACIAS")
    print(" Cidades a serem analisadas:")
    for i, city in enumerate(CITIES, 1):
        print(f"  {i}. {city}")

    print("\n Buscando apenas delegacias (amenity: police)")
    print("=" * 50)

    start_time = time.time()
    df = analyzer.analyze_all_cities(CITIES)
    analyzer.plot_results()
    end_time = time.time()

    print(f"\n Análise concluída em {end_time - start_time:.2f} segundos!")
    print("    Relatório salvo em: /content/results/comparative_analysis.csv")
    print("    Gráficos salvos em: /content/results/graphs/")

    # Mostrar links para download no Colab
    print("\n Para baixar os resultados:")
    print("   from google.colab import files")
    print("   files.download('/content/results/comparative_analysis.csv')")
    print("   !zip -r /content/results.zip /content/results/")
    print("   files.download('/content/results.zip')")

 Configurando ambiente...
 INICIANDO ANÁLISE DE MST ENTRE DELEGACIAS
 Cidades a serem analisadas:
  1. Maceió, Alagoas, Brazil
  2. Salvador, Bahia, Brazil
  3. Fortaleza, Ceará, Brazil
  4. São Luís, Maranhão, Brazil
  5. João Pessoa, Paraíba, Brazil
  6. Recife, Pernambuco, Brazil
  7. Teresina, Piauí, Brazil
  8. Natal, Rio Grande do Norte, Brazil
  9. Aracaju, Sergipe, Brazil

 Buscando apenas delegacias (amenity: police)
INICIANDO ANÁLISE COMPLETA

📋 [1/9] Próxima cidade: Maceió, Alagoas, Brazil

🏙️  Processando Maceió, Alagoas, Brazil
1. Baixando grafo viário...
 Grafo baixado: 16064 nós, 39603 arestas
2.  Buscando delegacias...
Buscando delegacias em Maceió...
 27 delegacias encontradas
 Primeiras coordenadas: [(-9.6562285, -35.711526), (-9.6244851, -35.7388587), (-9.657386, -35.7559092)]
3. Encontrando nós mais próximos...
  Encontrando nós para 27 delegacias...
 27 nós únicos encontrados
4.  Projetando grafo para UTM...
5. Calculando rotas A* entre delegacias...
 Total de pare

Rotas Maceió: 100%|██████████| 351/351 [00:08<00:00, 39.35it/s]


 Rotas calculadas: 351 (falhas: 0)
6. MST calculada: 75.38 km, 26 rotas
Maceió processada com sucesso!

📋 [2/9] Próxima cidade: Salvador, Bahia, Brazil

🏙️  Processando Salvador, Bahia, Brazil
1. Baixando grafo viário...
 Grafo baixado: 28213 nós, 64798 arestas
2.  Buscando delegacias...
Buscando delegacias em Salvador...
 83 delegacias encontradas
 Primeiras coordenadas: [(-13.0073412, -38.505073), (-12.994457, -38.459707), (-12.926778, -38.4529509)]
3. Encontrando nós mais próximos...
  Encontrando nós para 83 delegacias...
 66 nós únicos encontrados
 17 nós duplicados removidos
4.  Projetando grafo para UTM...
5. Calculando rotas A* entre delegacias...
 Total de pares a calcular: 2145


Rotas Salvador: 100%|██████████| 2145/2145 [01:15<00:00, 28.34it/s]


 Rotas calculadas: 2087 (falhas: 58)
6. MST calculada: 139.08 km, 65 rotas
Salvador processada com sucesso!

📋 [3/9] Próxima cidade: Fortaleza, Ceará, Brazil

🏙️  Processando Fortaleza, Ceará, Brazil
1. Baixando grafo viário...
 Grafo baixado: 36750 nós, 96222 arestas
2.  Buscando delegacias...
Buscando delegacias em Fortaleza...
 79 delegacias encontradas
 Primeiras coordenadas: [(-3.7089995, -38.4691259), (-3.8142956, -38.5693326), (-3.8264887, -38.5588803)]
3. Encontrando nós mais próximos...
  Encontrando nós para 79 delegacias...
 74 nós únicos encontrados
 5 nós duplicados removidos
4.  Projetando grafo para UTM...
5. Calculando rotas A* entre delegacias...
 Total de pares a calcular: 2701


...

 Rotas calculadas: 2415 (falhas: 0)
6. MST calculada: 74.81 km, 69 rotas
Natal processada com sucesso!

📋 [9/9] Próxima cidade: Aracaju, Sergipe, Brazil

🏙️  Processando Aracaju, Sergipe, Brazil
1. Baixando grafo viário...
 Grafo baixado: 11714 nós, 29704 arestas
2.  Buscando delegacias...
Buscando delegacias em Aracaju...
 36 delegacias encontradas
 Primeiras coordenadas: [(-10.9468619, -37.0524954), (-10.9365292, -37.0613026), (-10.9146514, -37.0810664)]
3. Encontrando nós mais próximos...
  Encontrando nós para 36 delegacias...
 36 nós únicos encontrados
4.  Projetando grafo para UTM...
5. Calculando rotas A* entre delegacias...
 Total de pares a calcular: 630


Rotas Aracaju: 100%|██████████| 630/630 [00:15<00:00, 39.82it/s]

 Rotas calculadas: 579 (falhas: 51)
6. MST calculada: 64.90 km, 34 rotas
Aracaju processada com sucesso!

RESUMO: 9 de 9 cidades processadas com sucesso

 RELATÓRIO COMPARATIVO - MST ENTRE DELEGACIAS


,Cidade,Total MST (km),Delegacias,Nós Únicos,Arestas na MST,Média por Aresta (km),Média por Delegacia (km)
0,Maceió,75.38,27,27,26,2.90,2.79
1,Salvador,139.08,83,66,65,2.14,1.68
2,Fortaleza,132.12,79,74,73,1.81,1.67
3,São Luís,111.57,41,39,38,2.94,2.72
4,João Pessoa,52.29,25,23,22,2.38,2.09
5,Recife,74.51,57,52,51,1.46,1.31
6,Teresina,101.11,38,35,34,2.97,2.66
7,Natal,74.81,100,70,69,1.08,0.75
8,Aracaju,64.90,36,36,34,1.91,1.80



 ESTATÍSTICAS GERAIS:
    Distância total MST: 825.77 km
    Média por cidade: 91.75 km
    Maior MST: Salvador (139.08 km)
    Menor MST: João Pessoa (52.29 km)

 Gerando gráficos para 9 cidades...
  Plotando Maceió...
 Gráfico salvo: /content/results/graphs/mst_maceió.png
  Plotando Salvador...
 Gráfico salvo: /content/results/graphs/mst_salvador.png
  Plotando Fortaleza...
 Gráfico salvo: /content/results/graphs/mst_fortaleza.png
  Plotando São Luís...
 Gráfico salvo: /content/results/graphs/mst_são_luís.png
  Plotando João Pessoa...
 Gráfico salvo: /content/results/graphs/mst_joão_pessoa.png
  Plotando Recife...
 Gráfico salvo: /content/results/graphs/mst_recife.png
  Plotando Teresina...
 Gráfico salvo: /content/results/graphs/mst_teresina.png
  Plotando Natal...
 Gráfico salvo: /content/results/graphs/mst_natal.png
  Plotando Aracaju...
 Gráfico salvo: /content/results/graphs/mst_aracaju.png

 Análise concluída em 725.21 segundos!
    Relatório salvo em: /content/results/compara

### Análise dos Resultados



#### Resultados Principais

##### Cidades Mais Eficientes:



*   Natal: Destaque absoluto com 0.75 km/delegacia - conecta 100 delegacias com apenas 74.81 km
*   Recife: Excelente eficiência (1.31 km/delegacia) com alta densidade (57 delegacias)
*   João Pessoa: Menor infraestrutura total (52.29 km) para 25 delegacias



##### Cidades com Maior Desafio:



*   Salvador: Maior extensão total (139.08 km) devido à dispersão geográfica

*   Teresina: Maior média por aresta (2.97 km) indicando conectividade desafiadora

*   São Luís: Alta média por delegacia (2.72 km) sugerindo distribuição esparsa



#### Padrões Identificados

##### Eficiência por Tipo de Cidade:



*   Cidades Compactas (Natal, Recife): < 1.5 km/delegacia

*   Cidades Médias (Fortaleza, Salvador): 1.6-1.8 km/delegacia

*   Cidades Dispersas (Teresina, São Luís): > 2.6 km/delegacia

##### Fatores de Influência:



*   Densidade Urbana: Cidades mais compactas têm MSTs menores

*   Distribuição Espacial: Localização estratégica das delegacias é crucial

*   Topografia: Barreiras naturais aumentam distâncias reais

*   Planejamento: Cidades com planejamento integrado mostram melhor eficiência

#### Insights para Políticas Públicas

##### Insights para Políticas Públicas

*   Teresina e São Luís podem se beneficiar de novas delegacias em áreas estratégicas

*   Salvador precisa de estratégias para reduzir distâncias médias

##### Métricas de Referência:

*   Meta ideal: < 1.5 km por delegacia

*   Área de atenção: > 2.5 km por delegacia

#### Conclusão Geral

O método A* + MST provou ser eficaz para análise comparativa de infraestrutura de segurança, revelando diferenças significativas na eficiência espacial entre as cidades. Natal obteve os melhores resultados na análise, apresentando a maior eficiência na conexão entre delegacias, enquanto outras cidades demonstram oportunidades para otimização em seus planejamentos de segurança pública.